# AF2 controlled illumination robustness — paired three-seed confirmation
Jalankan hanya jika screening seed 42 PASS. Inference-only; test tetap terkunci.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import importlib, json, os, shutil, subprocess, sys, tarfile, time, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/af2-illumination-robustness'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root, require_project_artifact
REQ=('bundles/faruq-development-v3-grouped.tar','experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/weights/best.pt','experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed123/weights/best.pt','experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed2026/weights/best.pt','experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt','experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed123/weights/best.pt','experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed2026/weights/best.pt','experiments/faruq-v3-af2-illumination-v1/illumination_screen_seed42.json')
PROJECT=resolve_drive_project_root(required_relative_paths=REQ); ARCHIVE=require_project_artifact(PROJECT,REQ[0]); SCREEN=require_project_artifact(PROJECT,REQ[-1])
screen=json.loads(SCREEN.read_text()); assert screen['decision']=='PASS' and screen['confirmation_authorized'] is True, 'STOP: screening tidak PASS.'
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE,'r') as archive:
        members=[m for m in archive.getmembers() if m.name.endswith('/data.yaml') or m.name.endswith('/faruq_grouped_summary.json') or '/val/' in m.name]
        archive.extractall('/content',members=members,filter='data')
assert (DATA/'val/images').is_dir() and not (DATA/'test').exists()
OUTPUT=PROJECT/'experiments/faruq-v3-af2-illumination-v1'
print('GPU:',torch.cuda.get_device_name(0)); print('OUTPUT:',OUTPUT)


In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_illumination','--project-root',str(PROJECT),'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--output-root',str(OUTPUT),'--seeds','42','123','2026','--screen-summary',str(SCREEN),'--device','0']
log=OUTPUT/'confirmation_three_seed_run.log'; handle=log.open('a',encoding='utf-8')
print('MENJALANKAN:', ' '.join(command),flush=True); process=subprocess.Popen(command,cwd=REPO,stdout=handle,stderr=subprocess.STDOUT,text=True)
while process.poll() is None:
    reports=len(list((OUTPUT/'reports').rglob('*.json'))) if (OUTPUT/'reports').exists() else 0
    print(f'ILLUMINATION CONFIRM: {reports}/60 evaluasi tersedia | log={log}',flush=True); time.sleep(300)
handle.close(); code=process.wait()
if code: print('\n'.join(log.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'Konfirmasi gagal: {code}')


In [ ]:
import pandas as pd
from IPython.display import display
summary=json.loads((OUTPUT/'illumination_confirmation_three_seed.json').read_text())
display(pd.DataFrame(summary['aggregate']).style.format({'mean_robustness_advantage':'{:+.2%}','minimum_robustness_advantage':'{:+.2%}'}))
display(pd.DataFrame(summary['clean_rows']).style.format({'d0ft_macro_map50_95':'{:.2%}','af2_macro_map50_95':'{:.2%}'}))
print('CRITERIA:',summary['criteria']); print('DECISION:',summary['decision']); print('TEST:',summary['test_images_accessed']); print('LIMIT:',summary['claim_limit'])
print('Kirim aggregate, clean rows, criteria, decision, dan claim limit.')
